# Rolling Chunks

## Setup and Imports

In [195]:
def chunk_text(text, chunk_size=150, overlap=20):
    """Split text into overlapping word-level chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.split()) >= 50:  # drop tiny tail chunks
            chunks.append(chunk)
    return chunks

In [196]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import nltk


In [197]:
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/graceegeorge/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [198]:
nltk_resources = [
    'tokenizers/punkt', 
    'taggers/averaged_perceptron_tagger', 
    'corpora/stopwords', 
    'help/tagsets']


In [199]:
for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)

[nltk_data] Error loading taggers/averaged_perceptron_tagger: Package
[nltk_data]     'taggers/averaged_perceptron_tagger' not found in
[nltk_data]     index
[nltk_data] Error loading help/tagsets: Package 'help/tagsets' not
[nltk_data]     found in index


In [200]:
OHCO = ['doc_title', 'para_num', 'sentence_num', 'token_num']


In [201]:
OHCO_chunk = ['doc_title', 'chunk_id', 'token_id']

In [202]:
LIB = pd.read_csv('data/pg2591-LIB.csv').set_index(OHCO[:1])    
LIB.head()

,volume
doc_title,
THE GOLDEN BIRD,1
HANS IN LUCK,1
JORINDA AND JORINDEL,1
THE TRAVELLING MUSICIANS,1
OLD SULTAN,1


## Making Chunks

In [203]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna()
TOKENS=TOKENS.join(LIB)
TOKENS.head()

pos_tuple pos token_str  \
doc_title para_num sentence_num token_num                                 
ASHPUTTEL 0        0            0           ('The', 'DT')  DT       The   
                                1          ('wife', 'NN')  NN      wife   
                                2            ('of', 'IN')  IN        of   
                                3             ('a', 'DT')  DT         a   
                                4          ('rich', 'JJ')  JJ      rich   

                                          term_str pos_group  volume  
doc_title para_num sentence_num token_num                             
ASHPUTTEL 0        0            0              the        DT       1  
                                1             wife        NN       1  
                                2               of        IN       1  
                                3                a        DT       1  
                                4             rich        JJ       1

In [204]:
STORIES = TOKENS.groupby(['doc_title']).term_str.apply(lambda x: " ".join(map(str,x)))
STORIES.head()

doc_title
ASHPUTTEL                       the wife of a rich man fell sick and when she ...
BRIAR ROSE                      a king and queen once upon a time reigned in a...
CAT AND MOUSE IN PARTNERSHIP    a certain cat had made the acquaintance of a m...
CAT-SKIN                        there was once a king whose queen had hair of ...
CLEVER ELSIE                    there was once a man who had a daughter who wa...
Name: term_str, dtype: object

In [205]:
CHUNKS = STORIES.apply(chunk_text).apply(pd.Series).stack().to_frame('chunk_str')
CHUNKS.index.names = ['doc_title', 'chunk_id']
CHUNK_LIB = CHUNKS.join(LIB)
CHUNK_LIB

chunk_str  volume
doc_title chunk_id                                                           
ASHPUTTEL 0         the wife of a rich man fell sick and when she ...       1
          1         fair in face but foul at heart and it was now ...       1
          2         by the hearth among the ashes and as this of c...       1
          3         broke it off and brought it away and when he g...       1
          4         were asked to come so they called her up and s...       1
...                                                               ...     ...
TOM THUMB 15        away tom however was still not disheartened an...       1
          16        s content as soon as he had had enough he want...       1
          17        they saw a wolf was there you may well suppose...       1
          18        tommy free ah said the father what fears we ha...       1
          19        and then they fetched new clothes for him for ...       1

[784 rows x 2 columns]

In [206]:
VOLUME_CHUNKS = CHUNK_LIB.groupby('volume')
VOLUME_CHUNKS.head()


chunk_str  \
doc_title      chunk_id                                                      
ASHPUTTEL      0         the wife of a rich man fell sick and when she ...   
               1         fair in face but foul at heart and it was now ...   
               2         by the hearth among the ashes and as this of c...   
               3         broke it off and brought it away and when he g...   
               4         were asked to come so they called her up and s...   
DOCTOR KNOWALL 0         there was once upon a time a poor peasant call...   
               1         frontispiece in the second turn your cart and ...   
               2         to go with him and bring back the stolen money...   
               3         actually was so he was terrified and said to h...   
               4         no idea what to say and cried ah poor crabb wh...   

                         volume  
doc_title      chunk_id          
ASHPUTTEL      0              1  
               1              1  
               2              1  
               3              1  
               4              1  
DOCTOR KNOWALL 0              2  
               1              2  
               2              2  
               3              2  
               4              2

In [207]:
X = VOLUME_CHUNKS.get_group(2)
X

chunk_str  \
doc_title                    chunk_id                                                      
DOCTOR KNOWALL               0         there was once upon a time a poor peasant call...   
                             1         frontispiece in the second turn your cart and ...   
                             2         to go with him and bring back the stolen money...   
                             3         actually was so he was terrified and said to h...   
                             4         no idea what to say and cried ah poor crabb wh...   
...                                                                                  ...   
THE WILLOW-WREN AND THE BEAR 2         are honest people bear you will have to pay fo...   
                             3         war thus war was announced to the bear and all...   
                             4         all animals you shall be general and lead us g...   
                             5         with such a humming and whirring and swarming ...   
                             6         the battle then the king and queen flew home t...   

                                       volume  
doc_title                    chunk_id          
DOCTOR KNOWALL               0              2  
                             1              2  
                             2              2  
                             3              2  
                             4              2  
...                                       ...  
THE WILLOW-WREN AND THE BEAR 2              2  
                             3              2  
                             4              2  
                             5              2  
                             6              2  

[221 rows x 2 columns]

In [208]:
CHUNKS['chunk_str'][1]

/var/folders/p9/9vqlyg213v3b9h9zrksgz6z00000gn/T/ipykernel_10283/817063780.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  CHUNKS['chunk_str'][1]


'fair in face but foul at heart and it was now a sorry time for the poor little girl what does the goodfornothing want in the parlour said they they who would eat bread should first earn it away with the kitchenmaid then they took away her fine clothes and gave her an old grey frock to put on and laughed at her and turned her into the kitchen there she was forced to do hard work to rise early before daylight to bring the water to make the fire to cook and to wash besides that the sisters plagued her in all sorts of ways and laughed at her in the evening when she was tired she had no bed to lie down on but was made to lie by the hearth among the ashes and as this of course made her always dusty and dirty they called her'

## Parse Chunks

In [209]:
# token_pat = r"[\s',-]+"
# CHUNKED_TOKENS = CHUNKS['chunk_str'].str.split(token_pat, expand=True).stack()\
#     .to_frame('token_str')
# CHUNKED_TOKENS.index.names = ['doc_title', 'chunk_id', 'token_id']
# CHUNKED_TOKENS

In [210]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    CHUNKED_TOKENS = CHUNKS.chunk_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    CHUNKED_TOKENS = CHUNKS.chunk_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')

## POS Tag Chunks

In [211]:
OHCO_chunk

['doc_title', 'chunk_id', 'token_id']

In [216]:
CHUNKED_TOKENS['pos'] = CHUNKED_TOKENS.pos_tuple.apply(lambda x: x[1])
CHUNKED_TOKENS['token_str'] = CHUNKED_TOKENS.pos_tuple.apply(lambda x: x[0])
CHUNKED_TOKENS['term_str'] = CHUNKED_TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
CHUNKED_TOKENS['pos_group'] = CHUNKED_TOKENS.pos.str[:2]
CHUNKED_TOKENS.index.names = OHCO_chunk
CHUNKED_TOKENS

pos_tuple  pos token_str term_str pos_group
doc_title chunk_id token_id                                               
ASHPUTTEL 0        0           (the, DT)   DT       the      the        DT
                   1          (wife, NN)   NN      wife     wife        NN
                   2            (of, IN)   IN        of       of        IN
                   3             (a, DT)   DT         a        a        DT
                   4          (rich, JJ)   JJ      rich     rich        JJ
...                                  ...  ...       ...      ...       ...
TOM THUMB 19       66           (s, VBZ)  VBZ         s        s        VB
                   67           (no, DT)   DT        no       no        DT
                   68        (place, NN)   NN     place    place        NN
                   69         (like, IN)   IN      like     like        IN
                   70         (home, NN)   NN      home     home        NN

[115213 rows x 5 columns]

In [213]:
CHUNKS.to_csv('data/chunks.csv')

In [217]:
CHUNKED_TOKENS.to_csv('data/chunked_tokens.csv')